# LeetCode 515 — Find Largest Value in Each Tree Row bằng Python

Link đề: https://leetcode.com/problems/find-largest-value-in-each-tree-row/ · Độ khó: **Medium** · Chủ đề: Tree, Binary Tree, Breadth-First Search, Depth-First Search

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def largestValues(self, root: Optional[TreeNode]) -> List[int]:
        ...
```

Ghi chú: notebook tự dựng `TreeNode` và hàm `build_tree` dựng cây từ list kiểu LeetCode (`[1,3,2,5,3,null,9]`). Ví dụ 3 (cây rỗng) là do notebook tự đặt, đáp án đã kiểm chứng bằng code; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với mỗi **hàng (row)** của cây nhị phân (đánh số từ 0 từ gốc xuống), tìm **giá trị lớn nhất** trong hàng đó, và trả về list các giá trị lớn nhất theo thứ tự hàng.
2. Nắm các công cụ Python: định nghĩa `class TreeNode`, dựng cây từ list có `None`, `collections.deque`, vòng `for _ in range(len(queue))` để xử lý theo tầng, đệ quy DFS có tham số `depth`, `max()`, ghép hai list có độ dài khác nhau bằng `zip_longest`.
3. Đi từ **liệt kê mọi hàng rồi lấy `max` từng hàng** → **BFS một lượt, vừa duyệt vừa giữ max của hàng** → **DFS có `depth`, cập nhật `res[depth]`** → **DFS lặp bằng ngăn xếp** → **chia để trị: gộp hai list kết quả của cây con**.
4. Hiểu mấu chốt: bài chia theo **độ sâu** nên chỉ cần mỗi nút **biết độ sâu của mình**; BFS cho độ sâu "miễn phí" theo từng vòng, DFS thì truyền thêm `depth`.
5. Tránh các lỗi kinh điển: khởi tạo max bằng `0` (sai khi giá trị âm), quên trường hợp cây rỗng, `IndexError` vì chưa tạo ô cho hàng mới, trộn nút của hàng này với hàng sau khi BFS, và nhầm "max mỗi hàng" với "max toàn cây".

## 2. Đề bài & ví dụ

Cho nút gốc `root` của một cây nhị phân, hãy trả về **mảng các giá trị lớn nhất của từng hàng** trong cây (**đánh số từ 0**).

**Ví dụ 1**

- Input: `root = [1,3,2,5,3,null,9]`
- Output: `[1,3,9]`
- Giải thích: hàng 0 là `[1]` → `1`; hàng 1 là `[3, 2]` → `3`; hàng 2 là `[5, 3, 9]` → `9`.

**Ví dụ 2**

- Input: `root = [1,2,3]`
- Output: `[1,3]`

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `root = []`
- Output: `[]`
- Giải thích: cây rỗng không có hàng nào.

**Ràng buộc:**

- Số nút của cây nằm trong khoảng `[0, 10^4]` (**có thể bằng 0**).
- `-2^31 <= Node.val <= 2^31 - 1`

Chú ý: giá trị có thể **âm** nên không thể khởi tạo "max hiện tại" bằng `0`.

In [1]:
from typing import Optional, List
from collections import deque, defaultdict
from itertools import zip_longest
import random
import sys

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([1, 3, 2, 5, 3, None, 9], [1, 3, 9]),
    ([1, 2, 3], [1, 3]),
    ([], []),
]
for vals, expected in examples:
    assert tree_to_list(build_tree(vals)) == vals
    print(vals, "-> kỳ vọng", expected)

[1, 3, 2, 5, 3, None, 9] -> kỳ vọng [1, 3, 9]
[1, 2, 3] -> kỳ vọng [1, 3]
[] -> kỳ vọng []


## 3. Kiến thức nền

### 3.1. Nút cây, độ sâu và hàng (row)

Mỗi nút `TreeNode` có `val`, `left`, `right`. **Độ sâu** của một nút là số cạnh từ gốc xuống nút đó (gốc: `0`). Các nút **cùng độ sâu** tạo thành một **hàng** (cũng gọi là level/tầng). Cây `[1,3,2,5,3,None,9]`:

```
        1          hàng 0
      /   \
     3     2       hàng 1
    / \     \
   5   3     9     hàng 2
```

In [2]:
def show_levels(root):
    level, d = [root], 0
    while any(x is not None for x in level):
        print(f"hàng {d}:", [x.val if x else None for x in level])
        level = [c for x in level if x for c in (x.left, x.right)]
        d += 1

show_levels(build_tree([1, 3, 2, 5, 3, None, 9]))

hàng 0: [1]
hàng 1: [3, 2]
hàng 2: [5, 3, None, 9]


### 3.2. Dựng cây từ list kiểu LeetCode

List `[1,3,2,5,3,None,9]` liệt kê các nút **theo từng hàng, từ trái sang phải**; `None` là chỗ trống. `build_tree` dùng hàng đợi: lấy nút đang chờ, gán con trái rồi con phải từ list, rồi đưa các con thật vào hàng đợi. `tree_to_list` làm ngược lại — ta dùng nó để kiểm tra việc dựng đúng.

In [3]:
t = build_tree([1, 3, 2, 5, 3, None, 9])
print("gốc:", t.val, "| con trái:", t.left.val, "| con phải:", t.right.val)
print("con phải của nút 2:", t.right.right.val, "| con trái của nút 2:", t.right.left)
print(tree_to_list(t))
print("cây rỗng:", build_tree([]))

gốc: 1 | con trái: 3 | con phải: 2
con phải của nút 2: 9 | con trái của nút 2: None
[1, 3, 2, 5, 3, None, 9]
cây rỗng: None


### 3.3. `deque` và BFS theo tầng

`collections.deque` là hàng đợi hai đầu: `append` thêm cuối, `popleft` lấy đầu, cả hai `O(1)`. BFS duyệt cây theo từng hàng: lấy các nút của hàng hiện tại ra, thêm con của chúng vào hàng đợi.

Cách tách ranh giới giữa các hàng: ở đầu mỗi vòng ngoài, chốt `size = len(queue)` — đó chính là **số nút của hàng hiện tại**. Vòng `for _ in range(size)` xử lý đúng các nút đó; những nút con được thêm vào trong lúc lặp thuộc hàng kế tiếp và không bị lẫn vào.

In [4]:
def bfs_demo(root):
    queue = deque([root])
    level = 0
    while queue:
        size = len(queue)
        row = []
        for _ in range(size):
            node = queue.popleft()
            row.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        print("hàng", level, "(size =", size, "):", row)
        level += 1

bfs_demo(build_tree([1, 3, 2, 5, 3, None, 9]))

hàng 0 (size = 1 ): [1]
hàng 1 (size = 2 ): [3, 2]
hàng 2 (size = 3 ): [5, 3, 9]


### 3.4. DFS đệ quy với tham số `depth`

DFS (duyệt theo chiều sâu) đệ quy: thăm nút rồi gọi con trái và con phải. Truyền thêm `depth` (con thì `depth + 1`) để mỗi nút biết nó thuộc **hàng nào**. Ta có thể ghi kết quả của hàng `d` vào `res[d]`: nếu `d == len(res)` nghĩa là **lần đầu** tới hàng đó và phải `append`, ngược lại cập nhật `res[d]`.

In [5]:
def collect_depths(node, depth, res):
    if node is None:
        return
    if depth == len(res):             # hàng mới: tạo ô mới
        res.append([])
    res[depth].append(node.val)
    collect_depths(node.left, depth + 1, res)
    collect_depths(node.right, depth + 1, res)

rows = []
collect_depths(build_tree([1, 3, 2, 5, 3, None, 9]), 0, rows)
print(rows)

[[1], [3, 2], [5, 3, 9]]


### 3.5. `max()` và giá trị khởi tạo cho "lớn nhất"

`max(iterable)` trả về phần tử lớn nhất. Khi tự viết vòng lặp cập nhật max, **đừng khởi tạo bằng `0`** vì giá trị có thể âm. Hai cách an toàn: lấy **phần tử đầu tiên** làm max ban đầu, hoặc dùng `float('-inf')`.

In [6]:
vals = [-5, -3, -8]
print(max(vals))

best = 0                      # sai khi mọi số đều âm
for v in vals:
    best = max(best, v)
print("khởi tạo 0 ->", best)

best = float("-inf")          # đúng
for v in vals:
    best = max(best, v)
print("khởi tạo -inf ->", best)

-3
khởi tạo 0 -> 0
khởi tạo -inf -> -3


### 3.6. `zip_longest` — ghép hai list độ dài khác nhau

Ở cách 5 ta cần ghép "max từng vị trí" của hai list có thể **dài ngắn khác nhau**. `itertools.zip_longest(a, b, fillvalue=...)` ghép theo vị trí và **điền** giá trị `fillvalue` cho phần thiếu của list ngắn hơn.

In [7]:
a = [3, 7]
b = [5, 1, 9]
print(list(zip_longest(a, b)))                      # None cho chỗ thiếu
merged = [max(x, y) for x, y in zip_longest(a, b, fillvalue=float("-inf"))]
print(merged)

[(3, 5), (7, 1), (None, 9)]
[5, 7, 9]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Bài yêu cầu "mỗi hàng một số". Cách nghĩ tự nhiên nhất của con người: **viết các hàng ra giấy, rồi gạch đít số lớn nhất ở mỗi hàng**.

Với `[1,3,2,5,3,None,9]`:

| Hàng | Các nút từ trái sang phải | Lớn nhất |
|---|---|---|
| 0 | `[1]` | `1` |
| 1 | `[3, 2]` | `3` |
| 2 | `[5, 3, 9]` | `9` |

Kết quả `[1, 3, 9]`. Các nút `5` và `3` ở hàng 2 là con của `3`, còn `9` là con của `2`; **bài không quan tâm nút thuộc nhánh nào**, chỉ quan tâm độ sâu.

Một ví dụ có số âm: `[0,-1,-5,-3,-2,-9,-8]`? Hãy thử bằng tay: hàng 0 `[0]`, hàng 1 `[-1,-5]` → `-1`, hàng 2 `[-3,-2,-9,-8]` → `-2`. Kết quả `[0,-1,-2]` — chú ý `-2` là lớn nhất vì gần `0` hơn; nếu khởi tạo `0` ta sẽ trả sai `0`.

Câu hỏi thiết kế: **làm sao biết hai nút có cùng hàng?** — hoặc đi từng hàng (BFS), hoặc gắn nhãn độ sâu cho mỗi nút (DFS).

In [8]:
t = build_tree([0, -1, -5, -3, -2, -9, -8])
show_levels(t)
print("mỗi hàng:", [[0], [-1, -5], [-3, -2, -9, -8]], "-> max:", [0, -1, -2])

hàng 0: [0]
hàng 1: [-1, -5]
hàng 2: [-3, -2, -9, -8]
mỗi hàng: [[0], [-1, -5], [-3, -2, -9, -8]] -> max: [0, -1, -2]


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi hàng rồi lấy `max` từng hàng

**Ý tưởng:**

1. BFS theo tầng để tạo `levels`: list các list giá trị của từng hàng.
2. Với mỗi hàng, lấy `max(row)`.

Rất sát với cách con người làm (tính tay ở mục 4), nên khó sai.

In [9]:
def largest_levels(root):
    if root is None:
        return []
    levels = []
    queue = deque([root])
    while queue:
        row = []
        for _ in range(len(queue)):
            node = queue.popleft()
            row.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        levels.append(row)
    return [max(row) for row in levels]

for vals, expected in examples:
    got = largest_levels(build_tree(vals))
    print(vals, "->", got)
    assert got == expected

[1, 3, 2, 5, 3, None, 9] -> [1, 3, 9]
[1, 2, 3] -> [1, 3]
[] -> []


**Độ phức tạp.**

- **Time:** `O(n)` — mỗi nút vào/ra hàng đợi một lần; `max` mỗi hàng tốn tổng cộng `O(n)`.
- **Space:** `O(n)` — `levels` giữ toàn bộ giá trị.

**Nút thắt:** ta lưu cả hàng chỉ để lấy `max`; có thể giữ một biến `max` cho mỗi hàng thay vì lưu mọi giá trị.

## 6. Ý tưởng thuật toán / bước đột phá

**Giữ một con số cho mỗi hàng thay vì cả hàng.** Có ba cách làm điều đó:

1. **BFS:** khi xử lý một hàng, giữ biến `row_max`, mỗi nút lấy ra thì `row_max = max(row_max, node.val)`; hết hàng thì `res.append(row_max)`.
2. **DFS có `depth`:** `res[depth]` là max hiện tại của hàng `depth`. Gặp nút ở độ sâu `d`: nếu `d == len(res)` thì `res.append(node.val)`, ngược lại `res[d] = max(res[d], node.val)`. Thứ tự duyệt **không quan trọng** vì `max` có tính giao hoán/kết hợp.
3. **Chia để trị:** `largest(node) = [node.val] + merge_max(largest(left), largest(right))`, trong đó `merge_max` lấy max theo từng vị trí (list ngắn hơn thì giữ nguyên phần dư). Hàng `d` của cây gốc `node` là hợp của hàng `d - 1` của hai cây con.

Vì mỗi nút được thăm đúng một lần và mỗi lần chỉ làm phép `max` hằng số, cả ba đều `O(n)`.

## 7. Các cách tiếp theo

### Cách 2 — BFS một lượt, giữ `row_max` (cách chuẩn)

**Giải thích từng đoạn:**

1. Cây rỗng → trả `[]`.
2. `queue = deque([root])`; `res = []`.
3. Mỗi vòng ngoài: `size = len(queue)`; `row_max = -inf`; lặp `size` lần: `popleft`, cập nhật `row_max`, thêm con vào hàng đợi.
4. Sau vòng trong, `res.append(row_max)`.

In [10]:
def largest_bfs(root):
    if root is None:
        return []
    res = []
    queue = deque([root])
    while queue:
        row_max = float("-inf")
        for _ in range(len(queue)):             # đúng các nút của hàng hiện tại
            node = queue.popleft()
            row_max = max(row_max, node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        res.append(row_max)
    return res

for vals, expected in examples:
    got = largest_bfs(build_tree(vals))
    assert got == expected
print("OK cách 2 trên ví dụ")

OK cách 2 trên ví dụ


In [11]:
class Solution:
    def largestValues(self, root: Optional[TreeNode]) -> List[int]:
        if root is None:
            return []
        res = []
        queue = deque([root])
        while queue:
            row_max = float("-inf")
            for _ in range(len(queue)):
                node = queue.popleft()
                row_max = max(row_max, node.val)
                if node.left:
                    queue.append(node.left)
                if node.right:
                    queue.append(node.right)
            res.append(row_max)
        return res

print([Solution().largestValues(build_tree(v)) for v, _ in examples])

[[1, 3, 9], [1, 3], []]


**Độ phức tạp (cách 2):** Time `O(n)`; Space `O(w)` với `w` là độ rộng lớn nhất của cây (kể cả `res` thì `O(h + w)`, trong đó `h` hàng ⇒ `res` dài `h`).

### Cách 3 — DFS đệ quy với `depth`

**Giải thích từng đoạn:**

1. `res = []`; `dfs(node, depth)`.
2. Nút rỗng → thoát.
3. Nếu `depth == len(res)`: đây là lần đầu tới hàng này → `res.append(node.val)`. Ngược lại `res[depth] = max(res[depth], node.val)`.
4. Gọi `dfs(left, depth + 1)` và `dfs(right, depth + 1)`.

Thứ tự thăm **không** quan trọng với bài này (khác bài 513), vì ta chỉ lấy `max`.

In [12]:
def largest_dfs(root):
    res = []

    def dfs(node, depth):
        if node is None:
            return
        if depth == len(res):
            res.append(node.val)
        else:
            res[depth] = max(res[depth], node.val)
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)

    dfs(root, 0)
    return res

for vals, expected in examples:
    got = largest_dfs(build_tree(vals))
    assert got == expected
print("OK cách 3 trên ví dụ")

OK cách 3 trên ví dụ


In [13]:
class Solution:
    def largestValues(self, root: Optional[TreeNode]) -> List[int]:
        res = []

        def dfs(node: Optional[TreeNode], depth: int) -> None:
            if node is None:
                return
            if depth == len(res):
                res.append(node.val)
            else:
                res[depth] = max(res[depth], node.val)
            dfs(node.left, depth + 1)
            dfs(node.right, depth + 1)

        dfs(root, 0)
        return res

print([Solution().largestValues(build_tree(v)) for v, _ in examples])

[[1, 3, 9], [1, 3], []]


**Độ phức tạp (cách 3):** Time `O(n)`; Space `O(h)` cho ngăn xếp đệ quy (cộng `res` dài `h`). Với cây lệch `10^4` nút, đệ quy sâu `10^4`: trên máy cá nhân cần `sys.setrecursionlimit`, hoặc dùng cách 2/4.

### Cách 4 — DFS lặp bằng ngăn xếp `(node, depth)`

Cùng ý tưởng cách 3 nhưng không đệ quy: ngăn xếp chứa các cặp `(node, depth)`. Vì chỉ lấy `max` nên **thứ tự đẩy trái/phải không ảnh hưởng đáp án**.

In [14]:
def largest_dfs_iter(root):
    if root is None:
        return []
    res = []
    stack = [(root, 0)]
    while stack:
        node, depth = stack.pop()
        if depth == len(res):
            res.append(node.val)
        else:
            res[depth] = max(res[depth], node.val)
        if node.left:
            stack.append((node.left, depth + 1))
        if node.right:
            stack.append((node.right, depth + 1))
    return res

for vals, expected in examples:
    got = largest_dfs_iter(build_tree(vals))
    assert got == expected
print("OK cách 4 trên ví dụ")

OK cách 4 trên ví dụ


Có một chi tiết tinh tế: với ngăn xếp (và cả đệ quy), một hàng **sâu** có thể được "chạm" trước khi hàng **nông hơn** được thăm hết. Điều này **không** gây `IndexError` vì ta chỉ `append` khi `depth == len(res)`. Nhưng có thể `depth > len(res)` không? Không: để thăm nút ở độ sâu `d`, ta đã thăm cha của nó ở độ sâu `d - 1`, tức `len(res) >= d`. Vì vậy `depth <= len(res)` luôn đúng.

In [15]:
class Solution:
    def largestValues(self, root: Optional[TreeNode]) -> List[int]:
        if root is None:
            return []
        res = []
        stack = [(root, 0)]
        while stack:
            node, depth = stack.pop()
            if depth == len(res):
                res.append(node.val)
            else:
                res[depth] = max(res[depth], node.val)
            if node.left:
                stack.append((node.left, depth + 1))
            if node.right:
                stack.append((node.right, depth + 1))
        return res

print([Solution().largestValues(build_tree(v)) for v, _ in examples])

[[1, 3, 9], [1, 3], []]


**Độ phức tạp (cách 4):** Time `O(n)`; Space `O(h)` thường, tối đa `O(n)` (ngăn xếp có thể chứa nhiều nút anh em). Không còn nguy cơ `RecursionError`.

### Cách 5 — Chia để trị: gộp kết quả của hai cây con

**Ý tưởng khác hẳn:** mô tả đáp án của một cây bằng đáp án của hai cây con.

```
largest(None) = []
largest(node) = [node.val] + merge_max(largest(node.left), largest(node.right))
```

`merge_max(a, b)`: với mỗi vị trí `i`, lấy `max(a[i], b[i])`; nếu một list ngắn hơn thì phần dư của list dài hơn giữ nguyên.

Vì sao đúng: hàng `d` của cây gốc `node` (với `d ≥ 1`) = hàng `d - 1` của cây con trái **hợp** hàng `d - 1` của cây con phải. Max của hợp = max của hai max.

In [16]:
def merge_max(a, b):
    NEG = float("-inf")
    return [max(x, y) for x, y in zip_longest(a, b, fillvalue=NEG)]

print(merge_max([3, 7], [5, 1, 9]))      # [5, 7, 9]

def largest_dc(root):
    if root is None:
        return []
    return [root.val] + merge_max(largest_dc(root.left), largest_dc(root.right))

for vals, expected in examples:
    got = largest_dc(build_tree(vals))
    assert got == expected
print("OK cách 5 trên ví dụ")

[5, 7, 9]
OK cách 5 trên ví dụ


In [17]:
class Solution:
    def largestValues(self, root: Optional[TreeNode]) -> List[int]:
        def merge(a: List[int], b: List[int]) -> List[int]:
            out = [max(x, y) for x, y in zip(a, b)]      # phần chung
            longer = a if len(a) > len(b) else b
            return out + longer[len(out):]               # phần dư của list dài hơn

        def solve(node: Optional[TreeNode]) -> List[int]:
            if node is None:
                return []
            return [node.val] + merge(solve(node.left), solve(node.right))

        return solve(root)

print([Solution().largestValues(build_tree(v)) for v, _ in examples])

[[1, 3, 9], [1, 3], []]


**Độ phức tạp (cách 5):** mỗi lần `merge` tốn `O(min(len(a), len(b)) + phần dư)`, nên tổng chi phí trên cả cây vào khoảng `O(n)` đến `O(n · h)` tùy cách tính (mỗi nút của cây con "trả phí" khi hàng của nó được gộp lên mỗi tầng tổ tiên; với cây lệch thì mỗi lần gộp chỉ copy một list dài `h`). Space `O(h)` đệ quy cộng các list trung gian. Cách này **hay về ý tưởng** nhưng không phải cách nhanh nhất; dùng cách 2 để nộp.

## 8. TRACE từng bước

### 8.1. Trace cách 2 (BFS giữ `row_max`)

Bảng trạng thái:

| Cột | Ý nghĩa |
|---|---|
| `hàng` / `size` | số thứ tự hàng và số nút của hàng (chốt đầu vòng ngoài) |
| `lấy ra` | nút vừa `popleft` |
| `row_max` | max hiện tại của hàng |
| `queue` | hàng đợi sau khi thêm con |

In [18]:
def trace_bfs(root):
    if root is None:
        print("cây rỗng -> []")
        return []
    res = []
    queue = deque([root])
    level = 0
    while queue:
        size = len(queue)
        row_max = float("-inf")
        print(f"--- hàng {level}, size = {size}, queue = {[x.val for x in queue]}")
        for _ in range(size):
            node = queue.popleft()
            row_max = max(row_max, node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
            print(f"   lấy ra {node.val:>3} | row_max = {row_max:>3} | queue = {[x.val for x in queue]}")
        res.append(row_max)
        print(f"   => hàng {level} xong, res = {res}")
        level += 1
    return res

trace_bfs(build_tree([1, 3, 2, 5, 3, None, 9]))

--- hàng 0, size = 1, queue = [1]
   lấy ra   1 | row_max =   1 | queue = [3, 2]
   => hàng 0 xong, res = [1]
--- hàng 1, size = 2, queue = [3, 2]
   lấy ra   3 | row_max =   3 | queue = [2, 5, 3]
   lấy ra   2 | row_max =   3 | queue = [5, 3, 9]
   => hàng 1 xong, res = [1, 3]
--- hàng 2, size = 3, queue = [5, 3, 9]
   lấy ra   5 | row_max =   5 | queue = [3, 9]
   lấy ra   3 | row_max =   5 | queue = [9]
   lấy ra   9 | row_max =   9 | queue = []
   => hàng 2 xong, res = [1, 3, 9]


[1, 3, 9]

Quan sát: ở hàng 1, khi lấy `3` ra thì thêm `5, 3` vào `queue`; khi lấy `2` ra thì thêm `9`. Hàng 2 bắt đầu với `size = 3` đúng bằng ba nút `5, 3, 9` — vì ta chốt `size` **trước** khi thêm các con mới, nên các nút hàng sau không lẫn vào hàng hiện tại.

### 8.2. Trace cách 3 (DFS có `depth`)

Mỗi dòng in nút, độ sâu, và `res` sau khi cập nhật. Nhìn kỹ thấy thứ tự thăm của DFS "nhảy" giữa các hàng nhưng kết quả vẫn đúng.

In [19]:
def trace_dfs(root):
    res = []

    def dfs(node, depth):
        if node is None:
            return
        if depth == len(res):
            res.append(node.val)
            action = "hàng mới -> append"
        else:
            old = res[depth]
            res[depth] = max(res[depth], node.val)
            action = f"max({old}, {node.val}) = {res[depth]}"
        print("  " * depth + f"thăm {node.val} (hàng {depth}): {action}; res = {res}")
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)

    dfs(root, 0)
    return res

trace_dfs(build_tree([1, 3, 2, 5, 3, None, 9]))

thăm 1 (hàng 0): hàng mới -> append; res = [1]
  thăm 3 (hàng 1): hàng mới -> append; res = [1, 3]
    thăm 5 (hàng 2): hàng mới -> append; res = [1, 3, 5]
    thăm 3 (hàng 2): max(5, 3) = 5; res = [1, 3, 5]
  thăm 2 (hàng 1): max(3, 2) = 3; res = [1, 3, 5]
    thăm 9 (hàng 2): max(5, 9) = 9; res = [1, 3, 9]


[1, 3, 9]

### 8.3. Trace cách 5 (chia để trị)

Ta in kết quả của từng cây con theo thứ tự hậu tự để thấy `merge` hoạt động:

In [20]:
def trace_dc(root, depth=0):
    pad = "  " * depth
    if root is None:
        return []
    a = trace_dc(root.left, depth + 1)
    b = trace_dc(root.right, depth + 1)
    out = [root.val] + merge_max(a, b)
    print(f"{pad}nút {root.val}: trái={a} phải={b} -> [{root.val}] + merge = {out}")
    return out

print("kết quả:", trace_dc(build_tree([1, 3, 2, 5, 3, None, 9])))

    nút 5: trái=[] phải=[] -> [5] + merge = [5]
    nút 3: trái=[] phải=[] -> [3] + merge = [3]
  nút 3: trái=[5] phải=[3] -> [3] + merge = [3, 5]
    nút 9: trái=[] phải=[] -> [9] + merge = [9]
  nút 2: trái=[] phải=[9] -> [2] + merge = [2, 9]
nút 1: trái=[3, 5] phải=[2, 9] -> [1] + merge = [1, 3, 9]
kết quả: [1, 3, 9]


## 9. Code nộp LeetCode

Chọn **cách 2 (BFS giữ `row_max`)**: dễ hiểu, `O(n)`, không đệ quy. Đây là code đúng class/method/signature (trên LeetCode `TreeNode`, `Optional`, `List` có sẵn; ở đây ta import để chạy được tại máy):

In [21]:
from collections import deque
from typing import List, Optional


class Solution:
    def largestValues(self, root: Optional[TreeNode]) -> List[int]:
        if root is None:
            return []
        res = []
        queue = deque([root])
        while queue:
            row_max = float("-inf")
            for _ in range(len(queue)):
                node = queue.popleft()
                row_max = max(row_max, node.val)
                if node.left:
                    queue.append(node.left)
                if node.right:
                    queue.append(node.right)
            res.append(row_max)
        return res


for vals, expected in examples:
    got = Solution().largestValues(build_tree(vals))
    print(vals, "->", got)
    assert got == expected

[1, 3, 2, 5, 3, None, 9] -> [1, 3, 9]
[1, 2, 3] -> [1, 3]
[] -> []


## 10. Vì sao thuật toán đúng

**BFS theo tầng (cách 2).** Khẳng định (bất biến vòng ngoài): đầu mỗi vòng ngoài thứ `d`, hàng đợi chứa **đúng các nút ở độ sâu `d`**, theo thứ tự từ trái sang phải. Ban đầu (`d = 0`) hàng đợi chỉ có gốc. Trong vòng ngoài, ta lấy ra đúng `size = len(queue)` nút — tất cả các nút hàng `d` — và với mỗi nút ta thêm con của nó vào cuối. Sau khi lấy ra hết hàng `d`, hàng đợi chứa đúng các con của chúng, tức các nút độ sâu `d + 1` (vì mỗi nút độ sâu `d + 1` có đúng một cha ở độ sâu `d`). Bất biến được bảo toàn. `row_max` là max của các giá trị đã lấy ra trong vòng đó = max của hàng `d`. Mỗi hàng không rỗng đều được duyệt → `res` có đúng `h` phần tử, theo thứ tự hàng tăng dần.

**Cây rỗng.** Không có hàng nào → trả `[]` (nhờ kiểm tra ở đầu hàm).

**Khởi tạo `row_max = -inf`.** Vì mỗi hàng có ít nhất một nút nên `row_max` luôn được cập nhật thành giá trị thật ngay nút đầu tiên; `-inf` không bao giờ lọt vào kết quả. Khởi tạo bằng `0` mới sai khi hàng toàn số âm.

**DFS (cách 3/4).** Mỗi nút ở độ sâu `d` được thăm đúng một lần và cập nhật `res[d]` bằng `max`. Vì `max` là phép giao hoán, kết hợp nên thứ tự thăm không ảnh hưởng. Điều kiện `depth == len(res)` hợp lệ vì khi thăm nút độ sâu `d` thì cha của nó (độ sâu `d - 1`) đã được thăm nên `len(res) >= d`.

**Chia để trị (cách 5).** Quy nạp theo chiều cao: nếu `solve(left)` và `solve(right)` đúng thì hàng `0` của `node` là `node.val`, hàng `d ≥ 1` là hợp của hàng `d - 1` hai con, max của hợp = `max` của hai max (nếu một con không có hàng đó thì chỉ lấy con còn lại).

## 11. Phân tích Time / Space Complexity

`n` là số nút, `h` số hàng (chiều cao), `w` độ rộng tối đa.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê mọi hàng, `max` từng hàng | `O(n)` | `O(n)` |
| 2. BFS giữ `row_max` | `O(n)` | `O(w)` |
| 3. DFS đệ quy + `depth` | `O(n)` | `O(h)` |
| 4. DFS lặp + `(node, depth)` | `O(n)` | `O(h)` đến `O(n)` |
| 5. Chia để trị + gộp list | `O(n)` đến `O(n·h)` | `O(h)` + list trung gian |

Mọi cách đều phải thăm **mọi nút** nên `O(n)` là cận dưới. Với `n = 10^4`, mọi cách đều chạy tức thì.

## 12. Bộ test

### 12.1. Ví dụ đề và edge case

Ta gom các cách vào dict `solvers`. **Oracle độc lập**: đánh số vị trí kiểu "heap" (con trái `2p`, con phải `2p + 1`, gốc `1`), suy ra độ sâu bằng `p.bit_length() - 1`, gom max theo độ sâu vào dict — không dùng hàng đợi theo tầng hay `depth` truyền qua đệ quy.

In [22]:
def oracle(root):
    best = {}
    if root is None:
        return []
    stack = [(root, 1)]
    while stack:
        node, pos = stack.pop()
        d = pos.bit_length() - 1                   # độ sâu từ chỉ số heap
        best[d] = max(best.get(d, node.val), node.val)
        if node.left:
            stack.append((node.left, 2 * pos))
        if node.right:
            stack.append((node.right, 2 * pos + 1))
    return [best[d] for d in sorted(best)]

solvers = {
    "levels": largest_levels,
    "bfs": largest_bfs,
    "dfs": largest_dfs,
    "dfs_iter": largest_dfs_iter,
    "dc": largest_dc,
    "leetcode": lambda r: Solution().largestValues(r),
}

for vals, expected in examples:
    root = build_tree(vals)
    assert oracle(root) == expected
    for name, fn in solvers.items():
        assert fn(root) == expected, (name, vals)
print("Ví dụ đề: OK cho cả 6 cách và oracle")

Ví dụ đề: OK cho cả 6 cách và oracle


In [23]:
edge_cases = [
    ([], [], "cây rỗng"),
    ([5], [5], "một nút"),
    ([-7], [-7], "một nút âm"),
    ([0], [0], "một nút 0"),
    ([2**31 - 1, -2**31], [2**31 - 1, -2**31], "biên giá trị"),
    ([1, None, 2, None, 3], [1, 2, 3], "lệch phải"),
    ([3, 2, None, 1], [3, 2, 1], "lệch trái"),
    ([0, -1, -5, -3, -2, -9, -8], [0, -1, -2], "toàn số âm ở các hàng dưới"),
    ([-1, -3, -2], [-1, -2], "mọi số đều âm"),
    ([1, 2, 3, 4, 5, 6, 7], [1, 3, 7], "cây đầy đủ"),
    ([7, 7, 7, 7, 7, 7, 7], [7, 7, 7], "mọi giá trị bằng nhau"),
    ([1, 5, 2, 9, None, None, 3], [1, 5, 9], "nhánh trái sâu hơn"),
    ([1, 2, 3, None, None, 4, 5, None, None, 6], [1, 3, 5, 6], "nhánh phải sâu hơn"),
]
for vals, expected, note in edge_cases:
    root = build_tree(vals)
    assert oracle(root) == expected, ("oracle", note)
    for name, fn in solvers.items():
        assert fn(root) == expected, (name, note, fn(root))
    print("OK:", note, "->", expected)

OK: cây rỗng -> []
OK: một nút -> [5]
OK: một nút âm -> [-7]
OK: một nút 0 -> [0]
OK: biên giá trị -> [2147483647, -2147483648]
OK: lệch phải -> [1, 2, 3]
OK: lệch trái -> [3, 2, 1]
OK: toàn số âm ở các hàng dưới -> [0, -1, -2]
OK: mọi số đều âm -> [-1, -2]
OK: cây đầy đủ -> [1, 3, 7]
OK: mọi giá trị bằng nhau -> [7, 7, 7]
OK: nhánh trái sâu hơn -> [1, 5, 9]
OK: nhánh phải sâu hơn -> [1, 3, 5, 6]


### 12.2. Phản ví dụ cho các cách làm sai

**Sai 1 — khởi tạo max bằng `0`.** **Sai 2 — trả về max toàn cây.** **Sai 3 — BFS không chốt `size`** (trộn nút của các hàng). **Sai 4 — quên xử lý cây rỗng.** **Sai 5 — DFS quên nhánh `depth == len(res)` (IndexError).**

In [24]:
def wrong_zero_init(root):
    if root is None:
        return []
    res, queue = [], deque([root])
    while queue:
        row_max = 0                                  # SAI: sai khi cả hàng đều âm
        for _ in range(len(queue)):
            node = queue.popleft()
            row_max = max(row_max, node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        res.append(row_max)
    return res

def wrong_global_max(root):
    m = max(oracle_vals(root))
    return [m]                                       # SAI: một số cho cả cây

def oracle_vals(root):
    out, stack = [], [root]
    while stack:
        x = stack.pop()
        if x:
            out.append(x.val)
            stack += [x.left, x.right]
    return out

def wrong_no_size(root):
    if root is None:
        return []
    res, queue = [], deque([root])
    while queue:
        row_max = float("-inf")
        while queue:                                 # SAI: lấy HẾT hàng đợi nên cả cây thành "một hàng"
            node = queue.popleft()
            row_max = max(row_max, node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        res.append(row_max)
    return res

def wrong_none_root(root):
    res, queue = [], deque([root])
    while queue:
        row_max = float("-inf")
        for _ in range(len(queue)):
            node = queue.popleft()
            row_max = max(row_max, node.val)         # AttributeError khi root là None
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        res.append(row_max)
    return res

def wrong_dfs_index(root):
    res = []
    def dfs(node, depth):
        if node is None:
            return
        res[depth] = max(res[depth], node.val)       # SAI: res chưa có ô cho hàng mới
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)
    dfs(root, 0)
    return res

test_trees = [v for v, _ in examples] + [v for v, _, _ in edge_cases]
for fn in (wrong_zero_init, wrong_global_max, wrong_no_size, wrong_none_root, wrong_dfs_index):
    bad = 0
    for v in test_trees:
        try:
            if fn(build_tree(v)) != oracle(build_tree(v)):
                bad += 1
        except Exception:
            bad += 1
    print(fn.__name__, "sai/lỗi trên", bad, "test")
    assert bad

wrong_zero_init sai/lỗi trên 4 test
wrong_global_max sai/lỗi trên 13 test
wrong_no_size sai/lỗi trên 11 test
wrong_none_root sai/lỗi trên 2 test
wrong_dfs_index sai/lỗi trên 14 test


### 12.3. Đối chiếu ngẫu nhiên (seed cố định) đúng ràng buộc

Sinh cây ngẫu nhiên `0 <= n <= 100` (cả cây rỗng), giá trị trong `[-2^31, 2^31 - 1]` hoặc khoảng nhỏ chứa số âm, so mọi cách với oracle.

In [25]:
def random_tree_list(rng, n, lo, hi):
    """Sinh cây ngẫu nhiên n nút (n >= 1) với giá trị trong [lo, hi], trả về list kiểu LeetCode."""
    root = TreeNode(rng.randint(lo, hi))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        k = rng.randrange(len(slots))
        slots[k], slots[-1] = slots[-1], slots[k]
        parent, side = slots.pop()
        child = TreeNode(rng.randint(lo, hi))
        setattr(parent, side, child)
        slots.append((child, "left"))
        slots.append((child, "right"))
    return tree_to_list(root)


rng = random.Random(515)
checked = 0
for _ in range(1500):
    n = rng.randint(0, 100)
    lo, hi = rng.choice([(-3, 3), (-1000, 1000), (-2**31, 2**31 - 1)])
    vals = random_tree_list(rng, n, lo, hi) if n else []
    root = build_tree(vals)
    want = oracle(root)
    for name, fn in solvers.items():
        assert fn(root) == want, (name, vals)
    checked += 1
print("đã đối chiếu", checked, "cây ngẫu nhiên (kể cả rỗng): 6 cách khớp oracle")

đã đối chiếu 1500 cây ngẫu nhiên (kể cả rỗng): 6 cách khớp oracle


In [26]:
# Cây lệch 10^4 nút (tối đa theo ràng buộc) và cây "đầy" ~10^4 nút
def chain_tree(n, side):
    root = TreeNode(0)
    cur = root
    for i in range(1, n):
        nxt = TreeNode(-i)
        if side == "left":
            cur.left = nxt
        else:
            cur.right = nxt
        cur = nxt
    return root

n = 10000
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(50000)
for side in ("left", "right"):
    root = chain_tree(n, side)
    want = [-i for i in range(n)]
    assert oracle(root) == want
    for name, fn in solvers.items():
        assert fn(root) == want, (side, name)
    print(side, "lệch 10^4 nút: OK")

full = build_tree(list(range(1, 2**13)))        # 8191 nút, đầy đủ
want = oracle(full)
assert want == [2**(d + 1) - 1 for d in range(13)]
for name, fn in solvers.items():
    assert fn(full) == want, name
print("cây đầy đủ 8191 nút: OK; max mỗi hàng =", want[:4], "...")
sys.setrecursionlimit(old_limit)

left lệch 10^4 nút: OK


right lệch 10^4 nút: OK
cây đầy đủ 8191 nút: OK; max mỗi hàng = [1, 3, 7, 15] ...


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Khởi tạo max bằng `0`

**Code sai:**

In [27]:
class WrongSolution1:
    def largestValues(self, root):
        if root is None:
            return []
        res, queue = [], deque([root])
        while queue:
            row_max = 0                                   # sai khi cả hàng âm
            for _ in range(len(queue)):
                node = queue.popleft()
                row_max = max(row_max, node.val)
                if node.left:
                    queue.append(node.left)
                if node.right:
                    queue.append(node.right)
            res.append(row_max)
        return res

t = build_tree([-1, -3, -2])
print("sai :", WrongSolution1().largestValues(t))
print("đúng:", Solution().largestValues(t))

sai : [0, 0]
đúng: [-1, -2]


**Vì sao sai:** các hàng chỉ có số âm nhưng `max(0, âm)` luôn ra `0`. **Khởi tạo bằng `-inf`** (hoặc lấy giá trị của nút đầu tiên trong hàng).

### Lỗi 2: Quên xử lý cây rỗng

In [28]:
class WrongSolution2:
    def largestValues(self, root):
        res, queue = [], deque([root])           # root có thể là None
        while queue:
            row_max = float("-inf")
            for _ in range(len(queue)):
                node = queue.popleft()
                row_max = max(row_max, node.val)  # None.val -> lỗi
                if node.left:
                    queue.append(node.left)
                if node.right:
                    queue.append(node.right)
            res.append(row_max)
        return res

try:
    WrongSolution2().largestValues(None)
except AttributeError as e:
    print("lỗi:", e)
print("đúng:", Solution().largestValues(None))

lỗi: 'NoneType' object has no attribute 'val'
đúng: []


**Vì sao sai:** ràng buộc cho phép **0 nút**. Phải `if root is None: return []` trước khi đưa vào hàng đợi.

### Lỗi 3: Không chốt `size` — nút của các hàng bị lẫn nhau

In [29]:
class WrongSolution3:
    def largestValues(self, root):
        if root is None:
            return []
        res, queue = [], deque([root])
        while queue:
            row_max = float("-inf")
            while queue:                              # SAI: lấy HẾT hàng đợi, kể cả con vừa thêm
                node = queue.popleft()
                row_max = max(row_max, node.val)
                if node.left:
                    queue.append(node.left)
                if node.right:
                    queue.append(node.right)
            res.append(row_max)
        return res

t = build_tree([1, 3, 2, 5, 3, None, 9])
print("sai :", WrongSolution3().largestValues(t))
print("đúng:", Solution().largestValues(t))

sai : [9]
đúng: [1, 3, 9]


**Vì sao sai:** vòng `while queue` bên trong chạy tới khi hàng đợi cạn, nên **cả cây** bị coi là một hàng. Phải dùng `for _ in range(len(queue))` để chỉ xử lý số nút đã chốt ở đầu hàng.

### Lỗi 4: DFS không tạo ô cho hàng mới (`IndexError`)

In [30]:
try:
    wrong_dfs_index(build_tree([1, 2, 3]))
except IndexError as e:
    print("IndexError:", e)
print("đúng:", largest_dfs(build_tree([1, 2, 3])))

IndexError: list index out of range
đúng: [1, 3]


**Vì sao sai:** lần đầu chạm tới hàng `d`, `res` còn ngắn (chưa có `res[d]`). Phải kiểm tra `if depth == len(res): res.append(node.val)` rồi mới `else` cập nhật `max`.

### Lỗi 5: Trả về max toàn cây thay vì mỗi hàng
Đã minh họa ở mục 12.2 (`wrong_global_max`). Đề cần **một số cho mỗi hàng**, độ dài kết quả bằng số hàng.

In [31]:
t = build_tree([1, 3, 2, 5, 3, None, 9])
print("sai :", wrong_global_max(t), "| đúng:", Solution().largestValues(t))

sai : [9] | đúng: [1, 3, 9]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mỗi hàng / mỗi tầng một giá trị"** → **BFS theo tầng** với `for _ in range(len(queue))`. Đây là khuôn mẫu cho: Level Order (102), Right Side View (199), Average of Levels (637), Largest Value in Each Row (515), Bottom-left Value (513)...
- Với BFS: nhớ **chốt `size` trước** khi vòng lặp thêm con vào hàng đợi.
- DFS cũng làm được nếu **gắn nhãn độ sâu** cho nút. Thứ tự duyệt chỉ quan trọng khi phép gộp **không giao hoán** (ví dụ bài 513 cần "trái trước"); với `max`, `sum`, `min` thì không.
- Luôn nghĩ tới **cây rỗng** (`root is None`) khi ràng buộc cho phép `0` nút.
- Với "giá trị lớn nhất" luôn khởi tạo bằng `-inf` hoặc phần tử đầu.
- Thấy giá trị có thể âm → đừng dùng `0` làm giá trị trung hòa cho `max`.

## 15. Biến thể

1. **Smallest value in each row:** đổi `max` thành `min` (khởi tạo `+inf`).
2. **Average of Levels (637):** giữ `sum` và `count` mỗi hàng.
3. **Right Side View (199):** lấy nút cuối mỗi hàng.
4. **Bottom Left Value (513):** phần tử đầu của hàng cuối.
5. **Maximum Level Sum (1161):** hàng có tổng lớn nhất (đánh số từ 1).
6. **N-ary tree:** thay `left/right` bằng danh sách `children`.

Ví dụ biến thể 1 và 2:

In [32]:
def smallest_values(root):
    if root is None:
        return []
    res, queue = [], deque([root])
    while queue:
        row_min = float("inf")
        for _ in range(len(queue)):
            node = queue.popleft()
            row_min = min(row_min, node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        res.append(row_min)
    return res

def average_of_levels(root):
    if root is None:
        return []
    res, queue = [], deque([root])
    while queue:
        size = len(queue)
        total = 0
        for _ in range(size):
            node = queue.popleft()
            total += node.val
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        res.append(total / size)
    return res

t = build_tree([1, 3, 2, 5, 3, None, 9])
print(smallest_values(t), average_of_levels(t))

[1, 2, 3] [1.0, 2.5, 5.666666666666667]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê mọi hàng + `max` | BFS lưu cả hàng | `O(n)` | `O(n)` | Dễ hiểu, đối chiếu |
| 2. BFS giữ `row_max` | Một biến max mỗi hàng | `O(n)` | `O(w)` | **Cách chuẩn để nộp** |
| 3. DFS đệ quy + `depth` | `res[depth] = max(...)` | `O(n)` | `O(h)` | Khi cây rộng (tiết kiệm bộ nhớ hơn BFS) |
| 4. DFS lặp (ngăn xếp) | Giống 3, không đệ quy | `O(n)` | `O(h)`~`O(n)` | Cây rất sâu |
| 5. Chia để trị | Gộp list của hai cây con | `O(n)`~`O(n·h)` | `O(h)` + list | Học ý tưởng "đáp án cha từ đáp án con" |

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [33]:
print(Solution().largestValues(build_tree([1, 3, 2, 5, 3, None, 9])))
print(Solution().largestValues(build_tree([4])))
print(Solution().largestValues(build_tree([])))

[1, 3, 9]
[4]
[]


**Bài 2 (tính tay).** Cây `[1,-5,-3,-8,-10]` (nút `-8` và `-10` là hai con của `-5`): liệt kê các hàng và tính max từng hàng. Gán vào `ans2` (đang là `None`).

In [34]:
ans2 = None      # TODO: list 3 số nguyên

**Bài 3 (điền TODO).** Viết `row_sums(root)` trả về list **tổng** của từng hàng. Ví dụ `[1,3,2,5,3,None,9]` -> `[1, 5, 17]`.

In [35]:
def row_sums(root):
    # TODO: BFS theo tầng, cộng giá trị của từng hàng
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_largest(root)` theo cách DFS có `depth` (cách 3).

In [36]:
def my_largest(root):
    # TODO
    pass

**Bài 5 (mini challenge).** Viết `max_level_sum(root)` (kiểu bài 1161): trả về **số thứ tự hàng (đánh số từ 1)** có tổng lớn nhất; nếu nhiều hàng bằng nhau thì trả về hàng **nhỏ nhất**. Ví dụ `[1,7,0,7,-8,None,None]` -> `2` (tổng các hàng: 1, 7, -1). Sau đó viết `row_ranges(root)` trả về list `(min, max)` của từng hàng.

In [37]:
def max_level_sum(root):
    # TODO
    pass

def row_ranges(root):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: viết ra các hàng. Cây rỗng thì sao?
- Bài 2: hàng 1 là `[-5, -3]`; hàng 2 chỉ có hai nút `-8, -10` (con của `-5`).
- Bài 3: giống cách 2 nhưng cộng thay vì lấy `max`.
- Bài 4: nhớ `if depth == len(res)`.
- Bài 5: dùng `row_sums`.

**Mức 2 (vừa):**
- Bài 2: `[1]`, `[-5, -3]`, `[-8, -10]` -> `[1, -3, -8]`.
- Bài 3: `total = 0` rồi `for _ in range(len(queue)): total += node.val`; `res.append(total)`.
- Bài 5: `sums = row_sums(root)`; hàng tốt nhất là `sums.index(max(sums)) + 1` (vì `index` trả về vị trí đầu tiên).

**Mức 3 (gần đáp án):**
- Bài 3: xem mục 7 (cách 2), thay `row_max` bằng `total`.
- Bài 4: xem cách 3.
- Bài 5: `row_ranges`: với mỗi hàng giữ cả `lo` và `hi`, `res.append((lo, hi))`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [38]:
# Bài 1
assert Solution().largestValues(build_tree([1, 3, 2, 5, 3, None, 9])) == [1, 3, 9]
assert Solution().largestValues(build_tree([4])) == [4]
assert Solution().largestValues(build_tree([])) == []

# Bài 2: các hàng [1], [-5, -3], [-8, -10]
ans2 = [1, -3, -8]
assert ans2 == Solution().largestValues(build_tree([1, -5, -3, -8, -10]))
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [39]:
# Bài 3
def row_sums(root):
    if root is None:
        return []
    res, queue = [], deque([root])
    while queue:
        total = 0
        for _ in range(len(queue)):
            node = queue.popleft()
            total += node.val
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        res.append(total)
    return res

assert row_sums(build_tree([1, 3, 2, 5, 3, None, 9])) == [1, 5, 17]
assert row_sums(build_tree([])) == []
assert row_sums(build_tree([-4])) == [-4]
print("Bài 3 OK")

Bài 3 OK


In [40]:
# Bài 4
def my_largest(root):
    res = []
    def dfs(node, depth):
        if node is None:
            return
        if depth == len(res):
            res.append(node.val)
        else:
            res[depth] = max(res[depth], node.val)
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)
    dfs(root, 0)
    return res

rng = random.Random(4)
for _ in range(200):
    n = rng.randint(0, 40)
    root = build_tree(random_tree_list(rng, n, -50, 50) if n else [])
    assert my_largest(root) == oracle(root)
print("Bài 4 OK")

Bài 4 OK


In [41]:
# Bài 5
def max_level_sum(root):
    sums = row_sums(root)
    return sums.index(max(sums)) + 1

def row_ranges(root):
    if root is None:
        return []
    res, queue = [], deque([root])
    while queue:
        lo, hi = float("inf"), float("-inf")
        for _ in range(len(queue)):
            node = queue.popleft()
            lo, hi = min(lo, node.val), max(hi, node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        res.append((lo, hi))
    return res

assert max_level_sum(build_tree([1, 7, 0, 7, -8, None, None])) == 2
assert max_level_sum(build_tree([989, None, 10250, 98693, -89388, None, None, None, -32127])) == 2
assert max_level_sum(build_tree([5])) == 1
assert max_level_sum(build_tree([1, 1, 1])) == 2        # hàng 1 tổng 2 lớn hơn 1
assert row_ranges(build_tree([1, 3, 2, 5, 3, None, 9])) == [(1, 1), (2, 3), (3, 9)]
assert row_ranges(build_tree([])) == []
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template BFS theo tầng:**

```python
from collections import deque

class Solution:
    def largestValues(self, root):
        if not root:
            return []
        res = []
        queue = deque([root])
        while queue:
            row_max = float("-inf")
            for _ in range(len(queue)):          # chốt size của hàng hiện tại
                node = queue.popleft()
                row_max = max(row_max, node.val)
                if node.left:
                    queue.append(node.left)
                if node.right:
                    queue.append(node.right)
            res.append(row_max)
        return res
```

**Template DFS có `depth`:**

```python
def dfs(node, depth):
    if not node:
        return
    if depth == len(res):
        res.append(node.val)
    else:
        res[depth] = max(res[depth], node.val)
    dfs(node.left, depth + 1)
    dfs(node.right, depth + 1)
```

**Checklist trước khi submit:**

- [ ] `root is None` -> trả `[]`?
- [ ] `row_max` khởi tạo `-inf` (không phải `0`)?
- [ ] BFS: dùng `for _ in range(len(queue))` để chốt hàng?
- [ ] DFS: có nhánh `depth == len(res)` để tạo ô cho hàng mới?
- [ ] `res.append(...)` sau mỗi hàng (BFS) đúng một lần?
- [ ] Đã thử: cây rỗng, một nút, toàn số âm, cây lệch, biên giá trị `-2^31`, `2^31 - 1`?